In [33]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import random
import string

def caesar(text, shift):
    return ''.join(chr((ord(c) - 65 + shift) % 26 + 65) for c in text)

SEQ_LEN = 6          # Фиксированная длина фраз
VOCAB_SIZE = 26      # A-Z алфавит
SHIFT = 4            # Сдвиг
N_SAMPLES = 800

In [34]:
chars = string.ascii_uppercase
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}

original = [''.join(random.choices(chars, k=SEQ_LEN)) for _ in range(N_SAMPLES)]
encrypted = [caesar(w, SHIFT) for w in original]

X = torch.tensor([[char_to_idx[c] for c in w] for w in encrypted], dtype=torch.long)
Y = torch.tensor([[char_to_idx[c] for c in w] for w in original], dtype=torch.long)

dataset = data.TensorDataset(X, Y)
loader = data.DataLoader(dataset, batch_size=64, shuffle=True)

In [35]:
class SimpleRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB_SIZE, 16)
        self.rnn = nn.LSTM(16, 32, batch_first=True)
        self.fc = nn.Linear(32, VOCAB_SIZE)

    def forward(self, x):
        x = self.emb(x)
        out, _ = self.rnn(x)
        return self.fc(out)

model = SimpleRNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [36]:
for epoch in range(12):
    model.train()
    for batch_X, batch_Y in loader:
        optimizer.zero_grad()
        preds = model(batch_X)
        loss = criterion(preds.view(-1, VOCAB_SIZE), batch_Y.view(-1))
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 1 == 0:
        print(f"epoch {epoch+1:2d} | loss: {loss.item()}")

epoch  1 | loss: 2.4219954013824463
epoch  2 | loss: 0.7755919098854065
epoch  3 | loss: 0.1308123618364334
epoch  4 | loss: 0.03963505104184151
epoch  5 | loss: 0.018978197127580643
epoch  6 | loss: 0.0125051848590374
epoch  7 | loss: 0.009982003830373287
epoch  8 | loss: 0.007855003699660301
epoch  9 | loss: 0.006585883442312479
epoch 10 | loss: 0.005845778156071901
epoch 11 | loss: 0.004846649710088968
epoch 12 | loss: 0.004449669737368822


In [37]:
model.eval()
test_words = ["DOGCAT", "ABCDEF", "XYZABC", "PYTHON"]

with torch.no_grad():
    for w in test_words:
        enc = caesar(w, SHIFT)
        inp = torch.tensor([[char_to_idx[c] for c in enc]], dtype=torch.long)

        pred_idx = model(inp).argmax(dim=-1).squeeze().tolist()
        pred_str = ''.join(idx_to_char[i] for i in pred_idx)

        print(f"encoded: {enc} - predicted: {pred_str}")

encoded: HSKGEX - predicted: DOGCAT
encoded: EFGHIJ - predicted: ABCDEF
encoded: BCDEFG - predicted: XYZABC
encoded: TCXLSR - predicted: PYTHON
